# 01 - EDA of the Geometry Dataset

## Role of this notebook
This notebook opens the project from the spatial layer. Its objective is to audit the geometry dataset, understand the distribution of the environmental variables, and document the decisions that enable the hybrid labeling process in notebook 02.

## Questions it answers
1. What information each spatial cell actually contains.
2. Which columns are useful for building labels and which ones do not contribute.
3. What observed ranges appear for `UDI`, `SUN_HOURS`, and `RADIATION`.
4. Which methodological decisions emerge from the EDA for the rest of the pipeline.

## Substages
1. Dataset loading and verification.
2. Quality audit.
3. Distributional analysis.
4. Reading for modeling decisions.
5. Export of intermediate findings.


In [3]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Define the project root so the notebook keeps working
# even when it is run from Jupyter inside the notebooks/ folder.
ROOT = Path.cwd().resolve().parent
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

GEOMETRY_PATH = RAW_DIR / 'geometry_dataset_daylight.csv'
df = pd.read_csv(GEOMETRY_PATH)
df.head()

,TILE_ID,BUILDING_ORIENTATION,RATIO_N,RATIO_E,RATIO_S,RATIO_W,UDI,SUN_HOURS,RADIATION
0,TILE_000,0,0.0,0.5,0.2,0.2,90.575,306.5,58.590
1,TILE_001,0,0.0,0.5,0.2,0.2,84.250,316.0,74.436
2,TILE_002,0,0.0,0.5,0.2,0.2,72.440,381.0,93.034
3,TILE_003,0,0.0,0.5,0.2,0.2,65.040,361.0,107.593
4,TILE_004,0,0.0,0.5,0.2,0.2,57.320,443.0,118.516


## 1. General Audit
First, we confirm size, data types, and general statistics. This step prevents rules or models from being built on false assumptions.


In [4]:
print(f'Rows: {len(df):,}')
print(f'Columns: {len(df.columns)}')
print(df.dtypes)

display(df.describe(include='all').transpose())

Rows: 43,100
Columns: 9
TILE_ID                     str
BUILDING_ORIENTATION      int64
RATIO_N                 float64
RATIO_E                 float64
RATIO_S                 float64
RATIO_W                 float64
UDI                     float64
SUN_HOURS               float64
RADIATION               float64
dtype: object


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
TILE_ID,43100,100,TILE_000,431,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BUILDING_ORIENTATION,43100.0,NaN,NaN,NaN,135.313225,100.530308,0.0,90.0,180.0,270.0,270.0
RATIO_N,43100.0,NaN,NaN,NaN,0.37587,0.302925,0.0,0.2,0.5,0.8,0.8
RATIO_E,43100.0,NaN,NaN,NaN,0.375406,0.303346,0.0,0.0,0.5,0.8,0.8
RATIO_S,43100.0,NaN,NaN,NaN,0.375406,0.303346,0.0,0.0,0.5,0.8,0.8
RATIO_W,43100.0,NaN,NaN,NaN,0.375406,0.303346,0.0,0.0,0.5,0.8,0.8
UDI,43100.0,NaN,NaN,NaN,19.28254,22.062519,1.1,3.23,11.42,25.95,99.85
SUN_HOURS,43100.0,NaN,NaN,NaN,1140.519884,593.116095,44.5,672.0,1069.0,1543.0,3045.0
RADIATION,43100.0,NaN,NaN,NaN,261.704265,108.53377,19.99,176.081,257.8545,344.322,547.69


In [ ]:
# This table summarizes data quality and helps detect columns
# with null values, little variation, or possible typing issues.
missing = df.isna().sum().rename('missing_values')
duplicates = int(df.duplicated().sum())
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_values': missing,
    'missing_pct': (missing / len(df) * 100).round(3),
    'n_unique': df.nunique()
}).sort_index()

print(f'Duplicated rows: {duplicates}')
display(summary)
summary.to_csv(PROCESSED_DIR / 'geometry_data_quality.csv', index=True)

## 2. Core Variables for Labeling
The project needs to build an interpretable spatial class. For that purpose, it uses three environmental variables already available for each cell:

- `UDI`: useful daylight quality.
- `SUN_HOURS`: accumulated sun hours.
- `RADIATION`: accumulated solar load.

We do not use a single variable because each one captures a different dimension of the phenomenon.

### Important Clarification About Percentiles
This notebook calculates a broad list of percentiles, but here their role is exploratory. Not all of them are used later for classification.

- `p10`, `p90`, `p95`: help read extremes and tails.
- `p25`, `p50`, `p75`: support a classic quartile-based reading.
- `p33`, `p67`: help compare whether a discretization into thirds would make sense.
- `p20`, `p40`, `p60`, `p80`: are the percentiles finally reused in notebook 02 to build levels 1-5.

The idea is to separate two moments: first, explore the real shape of the dataset; then choose the operational cut points with a clear criterion.


In [ ]:
key_columns = ['UDI', 'SUN_HOURS', 'RADIATION']
# This broad list is used only for diagnostics.
# The operational cut points for the final labeling are defined later in 02 with p20, p40, p60, and p80.
percentiles = [0.1, 0.2, 0.25, 0.33, 0.4, 0.5, 0.6, 0.67, 0.75, 0.8, 0.9, 0.95]
quantile_table = pd.DataFrame({
    col: df[col].quantile(percentiles) for col in key_columns
})
quantile_table.index = [f'p{int(q * 100):02d}' for q in percentiles]
display(quantile_table)
quantile_table.to_csv(PROCESSED_DIR / 'geometry_percentiles.csv')

### Why Not All of Those Percentiles Are Ultimately Used for Labeling
If too many cut points were used at the same time, the labeling would be harder to explain and less stable. For that reason, notebook 02 simplifies the decision and uses only `p20`, `p40`, `p60`, and `p80` to build five comparable levels:

- level 1: very low
- level 2: low
- level 3: medium
- level 4: high
- level 5: very high

In other words: here we observe a lot; later we use less, but with justification.


## 3. Distributions
The histograms show whether each variable is balanced, skewed, or dominated by extremes. This matters because the rules in notebook 02 will be based on percentiles rather than arbitrary thresholds.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, column in zip(axes, key_columns):
    # KDE is used to see the general shape of the distribution, not only the bin counts.
    sns.histplot(df[column], kde=True, ax=ax, bins=40)
    ax.set_title(f'Distribucion de {column}')
plt.tight_layout()

### Reading the Distributions
These charts help explain how exposure is distributed in the dataset and why it is useful to discretize with percentiles later. The main conclusions are:

1. `UDI` is clearly skewed toward low values. This indicates that many cells are concentrated in modest levels of useful daylight and that a symmetric distribution should not be assumed.
2. `SUN_HOURS` and `RADIATION` show broad dispersion. This is good for the project because it means the dataset contains enough contrast to separate different spatial conditions.
3. The distributions do not appear normal. For that reason, the pipeline avoids defining classes from the mean and standard deviation, and instead prefers observed percentiles.
4. The high tails of `SUN_HOURS` and `RADIATION` are especially useful for identifying potentially overexposed cells, while the low concentration of `UDI` helps recognize zones with low useful daylight.

Overall, the distributions suggest that the problem should be solved with a relative reading of the dataset: the goal is not to impose a universal threshold, but to locate each cell within the real structure of the data.


In [ ]:
distribution_summary = pd.DataFrame({
    'mean': df[key_columns].mean(),
    'median': df[key_columns].median(),
    'std': df[key_columns].std(),
    'min': df[key_columns].min(),
    'max': df[key_columns].max(),
    'skew': df[key_columns].skew()
}).round(3)

display(distribution_summary)
distribution_summary.to_csv(PROCESSED_DIR / 'geometry_distribution_summary.csv')

### Methodological Implications of Stage 3
Based on these distributions, three decisions are made:

- `UDI`, `SUN_HOURS`, and `RADIATION` are kept for label construction because they have enough variation and describe different aspects of exposure.
- Invented absolute cut points are not appropriate because the variables have different biases and scales.
- Percentile-based discretization is a more stable way to transform these continuous variables into comparable levels.


## 4. Correlations and Redundancy Reading
Although the problem cannot be reduced to linear correlations, this map helps detect strong redundancies and justify why some variables may or may not be discarded.


In [ ]:
numeric_columns = [
    'BUILDING_ORIENTATION', 'RATIO_N', 'RATIO_E', 'RATIO_S', 'RATIO_W',
    'UDI', 'SUN_HOURS', 'RADIATION'
]
corr = df[numeric_columns].corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='YlGnBu', fmt='.2f')
plt.title('Correlation between numeric variables')
plt.tight_layout()
corr

### Reading the Correlation Matrix
The correlation matrix should not be read as an automatic rule for removing variables. Its value here is interpretive: it helps identify which relationships are expected, which ones are strong, and where variables have little discriminative usefulness.

The main conclusions are:

1. `SUN_HOURS` and `RADIATION` show a high positive correlation. This is consistent with the physical phenomenon: more sun hours usually imply a higher radiative load.
2. `UDI` appears negatively correlated with `SUN_HOURS` and `RADIATION` in this dataset. This suggests that the relationship between useful daylight and accumulated solar exposure is not trivial, reinforcing the idea that a single variable is not enough to describe spatial suitability.
3. `RELATIVE_HUMIDITY` returns `NaN` across the matrix because it does not vary. This is the clearest signal that it does not contribute discriminative power to the classifier.
4. The geometric proxies do not show extreme correlations with each other, but they do show some moderate associations with the environmental variables. This supports the project hypothesis: geometry does contain relevant information about the luminous condition of the cell.

In other words, the matrix does not say that any environmental variable is redundant; rather, it shows that they play different roles and that some geometric proxies do have predictive value.


In [ ]:
upper_triangle_mask = pd.DataFrame(
    np.triu(np.ones(corr.shape), k=1).astype(bool),
    index=corr.index,
    columns=corr.columns
)
correlation_pairs = (
    corr.where(upper_triangle_mask)
        .stack()
        .reset_index()
        .rename(columns={'level_0': 'feature_1', 'level_1': 'feature_2', 0: 'correlation'})
        .sort_values('correlation', key=lambda s: s.abs(), ascending=False)
)

display(correlation_pairs.head(12).round(3))
correlation_pairs.to_csv(PROCESSED_DIR / 'geometry_correlation_pairs.csv', index=False)

### Methodological Implications of Stage 4
The matrix leads to concrete decisions for the pipeline:

- `RELATIVE_HUMIDITY` can be discarded as a model feature because it lacks useful variation.
- `UDI`, `SUN_HOURS`, and `RADIATION` are kept together for labeling because, although related, they are not conceptually equivalent.
- The geometric variables are kept for training because their association with the environmental variables supports the idea of a geometry-based spatial classifier.
- Correlation is used to support dataset interpretation, not as a mechanical column-removal filter.


## 5. Reading Geometric Combinations
The later classifier does not learn from `UDI`, `SUN_HOURS`, or `RADIATION`, but from the geometric proxies available before or during spatial configuration. For that reason, it is useful to see how combinations of ratios and orientation are organized.


In [ ]:
feature_grid = (
    df.groupby(['BUILDING_ORIENTATION', 'RATIO_N', 'RATIO_E', 'RATIO_S', 'RATIO_W'])
      .agg(
          cell_count=('TILE_ID', 'count'),
          mean_udi=('UDI', 'mean'),
          mean_sun=('SUN_HOURS', 'mean'),
          mean_radiation=('RADIATION', 'mean')
      )
      .reset_index()
      .sort_values('cell_count', ascending=False)
)
display(feature_grid.head(10))
feature_grid.to_csv(PROCESSED_DIR / 'geometry_feature_grid_summary.csv', index=False)

## 6. Methodological Decisions Derived From the EDA
This section makes explicit how the analysis feeds the rest of the pipeline. The idea is that the notebook should not only show charts, but also leave traceable reasoning.


In [ ]:
constant_columns = [col for col in df.columns if df[col].nunique() <= 1]
print('Constant or near-constant candidates:', constant_columns)

conclusions = pd.Series({
    'recommended_target_features': 'BUILDING_ORIENTATION, RATIO_N, RATIO_E, RATIO_S, RATIO_W',
    'recommended_label_inputs': 'UDI, SUN_HOURS, RADIATION',
    'drop_candidate': 'RELATIVE_HUMIDITY' if 'RELATIVE_HUMIDITY' in constant_columns else 'none',
    'decision_1': 'The spatial classifier will use geometric proxies rather than simulated indicators as training features.',
    'decision_2': 'The labeling will be built with observed dataset percentiles, not arbitrary thresholds.',
    'decision_3': 'Relative humidity will be marked as a discard candidate because it appears constant across the dataset.'
})
display(conclusions.to_frame('value'))
conclusions.to_csv(PROCESSED_DIR / 'geometry_eda_conclusions.csv', header=['value'])